# Sleep Aggregate

### init config and `df_backup` load

In [ ]:
import os
import sys

%load_ext autoreload
%autoreload 2
import logging

import pandas as pd
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt


# Configure logging
logging.basicConfig(level=logging.INFO)

# If your current working directory is the notebooks directory, use this:
notebook_dir = os.getcwd()  # current working directory
src_path = os.path.abspath(os.path.join(notebook_dir, "..", "src"))
sys.path.append(src_path)

# Add the parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(notebook_dir, ".."))
sys.path.append(parent_dir)
import pickle
from server_config import (
    datapath,
    preprocessed_path_freezed,
    redcap_path,
    preprocessed_path,
)
from functions.preprocessing.aggregation import compute_sleep_sessions


In [ ]:
# backup_path = preprocessed_path_freezed + "/backup_data_passive_actual.feather"
backup_path = (
    "/sc-projects/sc-proj-cc15-preact/SP6/raw/backup_passive_recent.feather"  # new file
)
df_backup = pd.read_feather(backup_path)
print(df_backup.shape)
assert "local_start_time" in df_backup.columns
assert "local_end_time" in df_backup.columns

df_backup.head()

In [ ]:
# remove customer with two for_ids for simplicity 
# TODO investigate later
customer_with_two_for_ids = "OmAV"
df_backup = df_backup[df_backup["customer"] != customer_with_two_for_ids]

## Sleep

_inspired by @RADAR_

| **No.** | **Column Name** | **Description** | 
| --| -- | -- |
| 1 | `customer` | Unique identifier for the customer/user | 
| 2 | `sleep_session_id` | Unique identifier for the specific sleep session | 
| 3 | `startTimestamp` | UTC timestamp marking the start of sleep session | 
| 4 | `endTimestamp` | UTC timestamp marking the end of sleep session | 
| 5 | `local_start_time` | Local time string/timestamp for the start of sleep session| 
| 6 | `local_end_time` | Local time string/timestamp for the end of sleep session | 
| 7 | `sleep_session_duration` | Total duration of the sleep session (end - start)| 
| 8 | `SleepAwake_duration` | Total duration spent in the awake stage | 
| 9 | `SleepDeep_duration` | Total duration spent in the deep sleep stage | 
| 10 | ~~`SleepInBed_duration`~~ | ~~Total duration spent in bed~~ (DO NOT USE - it isn't recorded for initial participants) | 
| 11 | `SleepLight_duration` | Total duration spent in the light sleep stage | 
| 12 | `total_sleep_time` | Total sleep time, i.e., sum of all "non-awake" stages, in seconds | 
| 13 | `awakenings` | Number of episodes in which the participant is awake for more than 5 minutes | 
| 14 | `long_awakenings` | Number of long awakening episodes (longer than 30 minutes) | 
| 15 | `sleep_onset` | Timestamp for the onset of sleep | 
| 16 | `sleep_offset` | Timestamp for the offset of sleep | 
| 17 | `sleep_onset_hour` | Exact onset time (in hours local time as decimal number, e.g., 22.5 = 10:30 p.m.) of sleep record | 
| 18 | `sleep_offset_hour` | Exact offset time (in hours local time as decimal number, e.g., 7.25 = 7:15 a.m.) of sleep record | 
| 19 | `time_in_bed` | Total time in bed, i.e., sum of all detected stages (including awake stages), in seconds | 
| 20 | `time_out_of_bed` | Total time spent out of bed during the session | 
| 21 | `sleep_efficiency` | Percentage of total sleep time to time in bed | 
| 22 | `hypersomnia` | flag if total sleep time > 10 hours | 
| 23 | `insomnia` | flag if total sleep time < 6 hours and at least one awakening of more than 30 minutes | 
| 24 | `awake_pct` | Proportion of total time in bed spent awake | 
| 25 | `light_sleep_pct` | Proportion of total time in bed spent in light sleep stage | 
| 26 | `deep_sleep_pct` | Proportion of total time in bed spent in deep sleep stage | 
| 27 | `day` | Day of the sleep session (day of the wake-up time)| 
| 28 | `num_sessions_in_day` | Count of distinct sleep sessions recorded for this customer on this day |

In [ ]:
df_sleep_sessions = compute_sleep_sessions(df_backup)
# TODO test different max_gap_second values (see below)
# TODO include out_of_bed periods in the time_in_bed
# TODO awake in bed & out of bed pct
df_sleep_sessions.head()

### add `for_id` column

In [ ]:
# customer - for_id are unique mapping, extract it and apply to df_sleep_sessions
customer_to_for_id = df_backup.groupby("customer")["for_id"].first()
df_sleep_sessions = df_sleep_sessions.merge(
    customer_to_for_id.rename("for_id"),
    left_on="customer",
    right_index=True,
    how="left",
)

In [ ]:
# TODO delete this line after renaming is finished and `compute_sleep_sessions` works with "id"
df_sleep_sessions.rename(columns={"customer": "id"}, inplace=True)

In [ ]:
df_sleep_sessions.head()

more than 90% of the customer-days got only one session, but some have more, what to do later?
- only analyze one, longest sleep session
- aggregate sessions further by summing over relevant columns

### pick the longest sleep session

In [ ]:
# new df with just longest sleep session per customer per day
df_sleep_sessions_longest = df_sleep_sessions.loc[
    df_sleep_sessions.groupby(["id", "day"])["sleep_session_duration"].idxmax()
].reset_index(drop=True)
# df_sleep_sessions_longest.groupby(["customer", "day"]).size().value_counts()
# TODO save as pickle/parquet
df_sleep_sessions_longest

### aggregate further to whole days

In [ ]:
df_sleep_sessions.groupby(["id", "day"]).agg(
    {
        "sleep_session_duration": "sum",
        "total_sleep_time": "sum",
        "time_in_bed": "sum",
        "time_out_of_bed": "sum",
        "SleepAwake_duration": "sum",
        "SleepLight_duration": "sum",
        "SleepDeep_duration": "sum",
        "awakenings": "sum",
        "long_awakenings": "sum",
        "num_sessions_in_day": "first",  # same for all sessions in that day
    }
)

In [ ]:
# TODO check if they record naps

### WIP

#### compare different `max_gap_seconds` values

In [ ]:
# Collect data for all max gaps
results = []
duration_data = []
sleep_time_data = []

for max_gap in [30 * 60, 45 * 60, 60 * 60, 75 * 60, 90 * 60, 120 * 60]:  # in seconds
    df_temp = compute_sleep_sessions(df_backup, max_gap_seconds=max_gap)
    
    df_valcounts = df_temp.num_sessions_in_day.value_counts().reset_index()
    df_valcounts["corrected_count"] = df_valcounts["count"] / df_valcounts.num_sessions_in_day
    df_valcounts["max_gap_minutes"] = max_gap // 60
    
    results.append(df_valcounts)
    
    # Collect duration and sleep time data for PDFs
    df_duration = df_temp[["sleep_session_duration", "total_sleep_time"]].copy()
    df_duration["max_gap_minutes"] = max_gap // 60
    duration_data.append(df_duration)

# Combine all results
df_all_gaps = pd.concat(results, ignore_index=True)
df_all_durations = pd.concat(duration_data, ignore_index=True)


In [ ]:
# Prepare data for plotting with zeros for missing values
max_gaps = sorted(df_all_gaps["max_gap_minutes"].unique())
max_sessions = df_all_gaps["num_sessions_in_day"].max()

# Create a complete dataset with zeros for missing combinations
plot_data = []
for gap in max_gaps:
    for n_sessions in range(1, max_sessions + 1):
        mask = (df_all_gaps["max_gap_minutes"] == gap) & (df_all_gaps["num_sessions_in_day"] == n_sessions)
        if mask.any():
            count = df_all_gaps.loc[mask, "corrected_count"].values[0]
        else:
            count = 0
        plot_data.append({"max_gap_minutes": gap, "num_sessions_in_day": n_sessions, "corrected_count": count})

df_plot = pd.DataFrame(plot_data)

# Create the plot
plt.figure(figsize=(12, 6))
bar_width = 0.15
x = np.arange(1, max_sessions + 1)

for i, gap in enumerate(max_gaps):
    df_gap = df_plot[df_plot["max_gap_minutes"] == gap]
    plt.bar(x + i * bar_width, df_gap["corrected_count"], width=bar_width, label=f"{gap} min gap", alpha=0.8)

plt.xlabel("Number of Sessions in Day")
plt.ylabel("Corrected Count (Number of Days)")
plt.title("Effect of Max Gap on Sleep Session Aggregation")
plt.xticks(x + bar_width * 2, x)
plt.yscale("log")
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig("../tmp/sleepagg-max-gap-num-days.png")
plt.show()

In [ ]:
# Plot distribution of total sleep time
plt.figure(figsize=(12, 6))

for gap in max_gaps:
    df_gap = df_all_durations[df_all_durations["max_gap_minutes"] == gap]
    # Convert seconds to hours
    sleep_time_hours = df_gap["total_sleep_time"] / 3600
    
    # Plot histogram/PDF
    plt.hist(sleep_time_hours, bins=50, alpha=0.5, label=f"{gap} min gap", density=True)

plt.xlabel("Total Sleep Time (hours)")
plt.ylabel("Probability Density")
plt.title("Distribution of Total Sleep Time for Different Max Gap Values")
plt.legend()
plt.grid(True, alpha=0.3)
# plt.xlim(0, 20)
plt.yscale('log')
plt.tight_layout()
plt.savefig("../tmp/sleepagg-max-gap-total-sleep-time.png")
plt.show()


In [ ]:
# Plot distribution of sleep session duration
plt.figure(figsize=(12, 6))

for gap in max_gaps:
    df_gap = df_all_durations[df_all_durations["max_gap_minutes"] == gap]
    # Convert seconds to hours
    durations_hours = df_gap["sleep_session_duration"] / 3600
    
    # Plot histogram/PDF
    plt.hist(durations_hours, bins=50, alpha=0.5, label=f"{gap} min gap", density=True)

plt.xlabel("Sleep Session Duration (hours)")
plt.ylabel("Probability Density")
plt.title("Distribution of Sleep Session Duration for Different Max Gap Values")
plt.legend()
plt.grid(True, alpha=0.3)
# plt.xlim(0, 20)
plt.yscale('log')

plt.tight_layout()
plt.savefig("../tmp/sleepagg-max-gap-sleep-session-duration.png")
plt.show()
